##### Chat bot with Multiple Tools
https://docs.langchain.com/oss/python/integrations/tools
- create chat bot with tool capablities from tavily(web search), Arxiv, wikipedia search and some custom functions

In [ ]:
## arxiv tool
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper

In [6]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=500)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv.name

'arxiv'

In [7]:
arxiv.invoke("Attention is all you need")

'Published: 2021-05-06\nTitle: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet\nAuthors: Luke Melas-Kyriazi\nSummary: The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the attention layer even necessary? Specifi'

In [8]:
## Wikipedia tool
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=500)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name


'wikipedia'

In [9]:
wiki.invoke("HUNTER X HUNTER")

'Page: Hunter × Hunter\nSummary: Hunter × Hunter (pronounced "hunter hunter") is a Japanese manga series written and illustrated by Yoshihiro Togashi. It has been serialized in Shueisha\'s shōnen manga magazine Weekly Shōnen Jump since March 1998, although the manga has frequently gone on extended hiatuses since 2006. Its chapters have been collected in 38 tankōbon volumes as of September 2024. The story focuses on a young boy named Gon Freecss who discovers that his father, who left him at a young'

In [10]:
## Tavily
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.environ.get("TAVILY_API_KEY")

In [14]:
"""
from langchain_tavily import TavilySearch

tavily = TavilySearch(
    max_results=5,
    topic="general",
)
"""

from langchain_community.tools.tavily_search import TavilySearchResults
tavily = TavilySearchResults()

C:\Users\dennis joshua j\AppData\Local\Temp\ipykernel_39640\3720625410.py:11: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults()


In [15]:
tavily.invoke({"query": "What are the IPL standings in 2026"})

[{'title': 'Indian Premier League Standings - 2026 - ESPN',
  'url': 'https://www.espn.com/cricket/standings/series/8048/ipl',
  'content': '| Team | M | W | L | T | N/R | PT | NRR | FOR | Against |\n ---  ---  ---  ---  --- |\n| 1Punjab KingsPBKS | 6 | 5 | 0 | 0 | 1 | 11 | 1.42 | 1050/93.1 | 985/100.0 |\n| 2Royal Challengers BengaluruRCB | 6 | 4 | 2 | 0 | 0 | 8 | 1.171 | 1218/110.5 | 1157/117.5 |\n| 3Rajasthan RoyalsRR | 6 | 4 | 2 | 0 | 0 | 8 | 0.599 | 1004/101.1 | 1032/110.4 |\n| 4Sunrisers HyderabadSRH | 6 | 3 | 3 | 0 | 0 | 6 | 0.566 | 1212/120.0 | 1090/114.2 |\n| 5Delhi CapitalsDC | 5 | 3 | 2 | 0 | 0 | 6 | 0.31 | 886/95.1 | 900/100.0 |\n| 6Gujarat TitansGT | 6 | 3 | 3 | 0 | 0 | 6 | -0.821 | 1022/118.2 | 1127/119.1 |\n| 7Mumbai IndiansMI | 6 | 2 | 4 | 0 | 0 | 4 | 0.067 | 1125/110.1 | 1072/105.4 |\n| 8Chennai Super KingsCSK | 6 | 2 | 4 | 0 | 0 | 4 | -0.78 | 1131/120.0 | 1131/110.5 | [...] | 9Lucknow Super GiantsLSG | 6 | 2 | 4 | 0 | 0 | 4 | -1.173 | 993/119.5 | 1050/111.0 |\n| 10Kolk

In [16]:
tools = [arxiv, wiki, tavily]

In [ ]:
# bind llm with tools
from langchain_groq import ChatGroq

llm = ChatGroq(model="qwen/qwen3-32b")

llm_with_tools = llm.bind_tools(tools)

In [18]:
from langchain_core.messages import AIMessage, HumanMessage

tool_call = llm_with_tools.invoke(
    [HumanMessage(content=f"what is the recent IPL news")]
)
tool_call

AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for recent IPL news. IPL is the Indian Premier League, a cricket tournament. Since they want recent news, I need to check the latest updates. The tools available are arxiv, wikipedia, and tavily_search. Arxiv is for academic papers, which doesn\'t fit. Wikipedia might have general info but not the latest news. Tavily_search is for current events, so that\'s the right choice. I\'ll use tavily_search with the query "recent IPL news" to get the latest information.\n', 'tool_calls': [{'id': 'jjdtg65fq', 'function': {'arguments': '{"query":"recent IPL news"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 139, 'prompt_tokens': 399, 'total_tokens': 538, 'completion_time': 0.259931154, 'completion_tokens_details': {'reasoning_tokens': 109}, 'prompt_time': 0.0207323, 'prompt_tokens_details': None, 'queue_time': 0.052771283, 'total_time'

In [19]:
# checking the tool response

tool_call.tool_calls

[{'name': 'tavily_search_results_json',
  'args': {'query': 'recent IPL news'},
  'id': 'jjdtg65fq',
  'type': 'tool_call'}]

#### Create entire chatbot with the help of langgraph